# CarRacing Test Environment
## Notebook interactivo para probar la pista personalizada

Este notebook te permite elegir entre diferentes modos de prueba:
1. **Control Manual**: Tú controlas el carro
2. **Agente Automático**: El agente DQN controla el carro
3. **Comparación**: Ver ambos modos uno después del otro

In [3]:
# Importar librerías necesarias
import gymnasium as gym
import pygame
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets
from Utils.interpreters import gym_interpreter_3
from Examples.car_racing import *
import warnings
warnings.filterwarnings("ignore")

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


## 🎮 Modo 1: Control Manual
Controla el carro tú mismo usando las teclas de flecha

In [4]:
def run_manual_control():
    """
    Ejecuta el modo de control manual
    """
    print("🎮 Iniciando Control Manual...")
    print("Controles:")
    print("  ⬅️ Flecha Izquierda: Girar izquierda")
    print("  ➡️ Flecha Derecha: Girar derecha")
    print("  ⬆️ Flecha Arriba: Acelerar")
    print("  ⬇️ Flecha Abajo: Frenar")
    print("  ESC: Salir")
    print("\n🚗 ¡Disfruta conduciendo por tu pista personalizada!")
    
    # Crear entorno
    env = gym.make('CarRacing-v2', 
                   render_mode='human',
                   continuous=False,
                   lap_complete_percent=0.07,
                   domain_randomize=False)
    
    # Inicializar pygame
    pygame.init()
    
    # Mapeo de acciones
    action_map = {
        pygame.K_LEFT: 1,    # Girar izquierda
        pygame.K_RIGHT: 2,   # Girar derecha
        pygame.K_UP: 3,      # Acelerar
        pygame.K_DOWN: 4,    # Frenar
    }
    
    obs, info = env.reset()
    total_reward = 0
    done = False
    step_count = 0
    
    clock = pygame.time.Clock()
    
    try:
        while not done:
            action = 0  # Por defecto: no hacer nada
            
            # Manejar eventos de pygame
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    done = True
                elif event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_ESCAPE:
                        done = True
            
            # Verificar teclas presionadas
            keys = pygame.key.get_pressed()
            for key, mapped_action in action_map.items():
                if keys[key]:
                    action = mapped_action
                    break
            
            # Ejecutar acción
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            
            total_reward += reward
            step_count += 1
            
            # Renderizar
            env.render()
            clock.tick(30)  # 30 FPS
            
            # Mostrar progreso cada 100 pasos
            if step_count % 100 == 0:
                print(f"📊 Pasos: {step_count}, Recompensa: {total_reward:.2f}")
    
    except KeyboardInterrupt:
        print("\n⏹️ Interrumpido por el usuario")
    
    finally:
        print(f"\n🏁 ¡Episodio completado!")
        print(f"📈 Pasos totales: {step_count}")
        print(f"🏆 Recompensa total: {total_reward:.2f}")
        env.close()
        pygame.quit()
        return step_count, total_reward

# Botón para ejecutar control manual
manual_button = widgets.Button(
    description='🎮 Iniciar Control Manual',
    button_style='info',
    layout=widgets.Layout(width='300px', height='50px')
)

def on_manual_click(b):
    with manual_output:
        clear_output(wait=True)
        run_manual_control()

manual_button.on_click(on_manual_click)
manual_output = widgets.Output()

display(manual_button, manual_output)

Button(button_style='info', description='🎮 Iniciar Control Manual', layout=Layout(height='50px', width='300px'…

Output()

## 🤖 Modo 2: Agente Automático
Deja que el agente DQN controle el carro

In [ ]:
def run_agent_control(show_visual=True, load_trained=False):
    """
    Ejecuta el agente DQN
    """
    print("🤖 Iniciando Agente Automático...")
    
    if load_trained:
        print("📚 Cargando agente entrenado...")
    else:
        print("🆕 Usando agente sin entrenar...")
    
    # Configurar el entorno y agente
    from Utils.train import Trainer
    from Agents.agentsCS import DQN
    from Agents.deepQ import DeepQ
    from Agents.networks import FFN
    
    # Parámetros del agente
    parameters = {
        'numDims': 1,
        'nA': 5,
        'gamma': 0.99,
        'epsilon': None,
        'alpha': 0.001,
        'c': 100,
        'len_exp': 1000,
        'len_mini_batch': 32
    }
    
    # Crear red neuronal y agente
    network = FFN(1024, 5, [512, 256])
    Q = DeepQ(network)
    agent = DQN(parameters, Q)
    
    # Cargar modelo entrenado si se solicita
    if load_trained:
        try:
            agent.load(name="ffn_trained")
            print("✅ Modelo entrenado cargado exitosamente")
        except:
            print("⚠️ No se pudo cargar el modelo entrenado, usando agente sin entrenar")
    
    # Crear trainer
    interpreter = gym_interpreter_3()
    trainer = Trainer(
        env_name='CarRacing-v2',
        state_interpreter=interpreter,
        agent=agent,
        model_name='DQN',
        num_rounds=1,
        num_episodes=1
    )
    
    print("🚗 Ejecutando episodio...")
    
    # Ejecutar episodio
    step_count, total_reward = trainer.run(visual=show_visual, learn=False)
    
    print(f"\n🏁 ¡Episodio completado!")
    print(f"📈 Pasos totales: {step_count}")
    print(f"🏆 Recompensa total: {total_reward:.2f}")
    
    return step_count, total_reward

# Controles para el agente
visual_checkbox = widgets.Checkbox(
    value=True,
    description='Mostrar visualización',
    disabled=False
)

trained_checkbox = widgets.Checkbox(
    value=False,
    description='Usar agente entrenado',
    disabled=False
)

agent_button = widgets.Button(
    description='🤖 Iniciar Agente',
    button_style='success',
    layout=widgets.Layout(width='300px', height='50px')
)

def on_agent_click(b):
    with agent_output:
        clear_output(wait=True)
        run_agent_control(visual_checkbox.value, trained_checkbox.value)

agent_button.on_click(on_agent_click)
agent_output = widgets.Output()

display(widgets.VBox([visual_checkbox, trained_checkbox, agent_button]), agent_output)

## 🆚 Modo 3: Comparación
Ejecuta ambos modos y compara los resultados

In [ ]:
def run_comparison():
    """
    Ejecuta una comparación entre control manual y agente
    """
    print("🆚 Iniciando Modo Comparación...")
    print("\nPrimero ejecutaremos el agente automático, luego el control manual.")
    
    results = {}
    
    # Ejecutar agente primero
    print("\n" + "="*50)
    print("🤖 FASE 1: AGENTE AUTOMÁTICO")
    print("="*50)
    
    agent_steps, agent_reward = run_agent_control(show_visual=True, load_trained=False)
    results['agent'] = {'steps': agent_steps, 'reward': agent_reward}
    
    input("\n⏸️ Presiona ENTER para continuar con el control manual...")
    
    # Ejecutar control manual
    print("\n" + "="*50)
    print("🎮 FASE 2: CONTROL MANUAL")
    print("="*50)
    
    manual_steps, manual_reward = run_manual_control()
    results['manual'] = {'steps': manual_steps, 'reward': manual_reward}
    
    # Mostrar comparación
    print("\n" + "="*50)
    print("📊 RESULTADOS DE LA COMPARACIÓN")
    print("="*50)
    
    print(f"🤖 Agente Automático:")
    print(f"   📈 Pasos: {results['agent']['steps']}")
    print(f"   🏆 Recompensa: {results['agent']['reward']:.2f}")
    
    print(f"\n🎮 Control Manual:")
    print(f"   📈 Pasos: {results['manual']['steps']}")
    print(f"   🏆 Recompensa: {results['manual']['reward']:.2f}")
    
    # Determinar ganador
    if results['manual']['reward'] > results['agent']['reward']:
        print(f"\n🏅 ¡El control manual ganó por {results['manual']['reward'] - results['agent']['reward']:.2f} puntos!")
    elif results['agent']['reward'] > results['manual']['reward']:
        print(f"\n🏅 ¡El agente automático ganó por {results['agent']['reward'] - results['manual']['reward']:.2f} puntos!")
    else:
        print(f"\n🤝 ¡Empate! Ambos obtuvieron la misma recompensa.")
    
    # Crear gráfico de comparación
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Gráfico de pasos
    modes = ['Agente', 'Manual']
    steps = [results['agent']['steps'], results['manual']['steps']]
    colors = ['#ff7f0e', '#1f77b4']
    
    ax1.bar(modes, steps, color=colors)
    ax1.set_title('Pasos Totales')
    ax1.set_ylabel('Número de Pasos')
    
    # Gráfico de recompensas
    rewards = [results['agent']['reward'], results['manual']['reward']]
    
    ax2.bar(modes, rewards, color=colors)
    ax2.set_title('Recompensa Total')
    ax2.set_ylabel('Recompensa')
    
    plt.tight_layout()
    plt.show()
    
    return results

# Botón para comparación
comparison_button = widgets.Button(
    description='🆚 Iniciar Comparación',
    button_style='warning',
    layout=widgets.Layout(width='300px', height='50px')
)

def on_comparison_click(b):
    with comparison_output:
        clear_output(wait=True)
        run_comparison()

comparison_button.on_click(on_comparison_click)
comparison_output = widgets.Output()

display(comparison_button, comparison_output)

## 🏋️ Modo 4: Entrenar Agente
Entrena un nuevo agente en tu pista personalizada

In [ ]:
def train_agent(num_rounds=50):
    """
    Entrena el agente DQN
    """
    print(f"🏋️ Iniciando entrenamiento del agente por {num_rounds} rondas...")
    
    # Ejecutar entrenamiento
    from Examples.car_racing import train
    
    # Modificar temporalmente el número de rondas
    import Examples.car_racing as cr
    original_rounds = getattr(cr, 'num_rounds', 250)
    cr.num_rounds = num_rounds
    
    try:
        train()
        print(f"\n✅ ¡Entrenamiento completado!")
        print(f"💾 El modelo entrenado se ha guardado automáticamente.")
    except Exception as e:
        print(f"❌ Error durante el entrenamiento: {e}")
    finally:
        # Restaurar valor original
        cr.num_rounds = original_rounds

# Control para número de rondas de entrenamiento
rounds_slider = widgets.IntSlider(
    value=50,
    min=10,
    max=500,
    step=10,
    description='Rondas:',
    style={'description_width': 'initial'}
)

train_button = widgets.Button(
    description='🏋️ Entrenar Agente',
    button_style='danger',
    layout=widgets.Layout(width='300px', height='50px')
)

def on_train_click(b):
    with train_output:
        clear_output(wait=True)
        train_agent(rounds_slider.value)

train_button.on_click(on_train_click)
train_output = widgets.Output()

display(widgets.VBox([rounds_slider, train_button]), train_output)

## 📊 Análisis de Resultados
Espacio para analizar y visualizar los resultados de tus pruebas

In [ ]:
# Espacio para análisis personalizado
print("📊 Usa esta celda para analizar tus resultados")
print("Puedes agregar código aquí para:")
print("- Comparar diferentes configuraciones")
print("- Visualizar el progreso del entrenamiento")
print("- Analizar estrategias de conducción")
print("- Guardar y cargar resultados")

# Ejemplo de análisis
# results_history = []
# 
# def save_result(mode, steps, reward):
#     results_history.append({
#         'mode': mode,
#         'steps': steps,
#         'reward': reward,
#         'timestamp': datetime.now()
#     })
#
# def plot_history():
#     if results_history:
#         df = pd.DataFrame(results_history)
#         # Crear gráficos de análisis
#         pass